# Demo 01 : One agent and a growing tool list

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We load the Veenhaven Bus database, service manuals and past repair tickets.
The database contains depots, mechanics, buses, parts and work orders.

In [2]:
import csv
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

db = sqlite3.connect(data("demo-world/service.db"), check_same_thread=False)
db.row_factory = sqlite3.Row

SECTIONS = []
for path in sorted(Path(data_folder("demo-world/manuals")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]
TICKETS = [", ".join(f"{k}={v}" for k, v in t.items())
           for t in csv.DictReader(open(data("demo-world/tickets.csv"), encoding="utf-8"))]


def rows(sql, *args):
    return [dict(r) for r in db.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]


print("sections:", len(SECTIONS), " tickets:", len(TICKETS),
      " buses:", len(rows("SELECT * FROM buses")), " parts:", len(rows("SELECT * FROM parts")))

sections: 82  tickets: 43  buses: 10  parts: 13


We define eight tools for service desk lookups. Each is a small function over
the data ; the model reads only the name, the arguments and the docstring.

In [3]:
@tool
def bus_info(fleet_no: str) -> str:
    """Model, depot and firmware of one bus, by fleet number such as CE12-2021-0117."""
    return str(rows("SELECT * FROM buses WHERE fleet_no = ?", fleet_no.strip().upper()) or "no such bus")


@tool
def part_stock(part_no: str) -> str:
    """Name, units in stock, price and depot of one spare part, by part number such as CE-CF-02."""
    return str(rows("SELECT * FROM parts WHERE part_no = ?", part_no.strip().upper()) or "no such part")


@tool
def parts_for_model(model: str) -> str:
    """Every spare part for a bus model such as CE-12, with units in stock."""
    return str(rows("SELECT * FROM parts WHERE model = ?", model.strip().upper()))


@tool
def search_manuals(query: str) -> str:
    """Search the service manuals and bulletins. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

@tool
def mechanics_for_model(model: str) -> str:
    """Mechanics certified for a bus model such as CD-12, with their region."""
    return str(rows("SELECT name, region, certified_models FROM mechanics WHERE certified_models LIKE ?",
                    f"%{model.strip().upper()}%"))


@tool
def depot_info(depot: str) -> str:
    """Id, city and service contract of a depot, by name, city or id."""
    d = depot.strip()
    return str(rows("SELECT * FROM depots WHERE depot_id = ? OR name LIKE ? OR city LIKE ?",
                    d.upper(), f"%{d}%", f"%{d}%"))


@tool
def work_orders_for_bus(fleet_no: str) -> str:
    """Work orders for one bus : status, error code, summary and mechanic."""
    return str(rows("SELECT * FROM work_orders WHERE fleet_no = ?", fleet_no.strip().upper()) or "none")


@tool
def search_tickets(query: str) -> str:
    """Search past repair tickets by fleet number, error code or symptom. Returns the five best matches."""
    return "\n".join(best(TICKETS, query, 5))

CORE = [bus_info, part_stock, parts_for_model, mechanics_for_model, depot_info,
        work_orders_for_bus, search_manuals, search_tickets]
print("core tools:", [t.name for t in CORE])

core tools: ['bus_info', 'part_stock', 'parts_for_model', 'mechanics_for_model', 'depot_info', 'work_orders_for_bus', 'search_manuals', 'search_tickets']


A real company has many more systems : timetables, rosters, payroll,
procurement. When one agent serves the whole company, their tools land in the
same list. We add 52 such tools. None of them helps with a repair question,
and each answers that it has no records.

In [4]:
from langchain_core.tools import StructuredTool
from pydantic import create_model

OTHER = [
    ("timetable_for_route", "Departure times for a bus route on a given day.", "route"),
    ("route_detours", "Current detours and closed stops on a route.", "route"),
    ("stop_info", "Location, shelter and accessibility of a bus stop.", "stop"),
    ("passenger_counts", "Boardings per trip on a route last week.", "route"),
    ("fare_for_journey", "Adult fare between two stops.", "journey"),
    ("ticket_machine_status", "Faults reported by the ticket machine on board a bus.", "fleet_no"),
    ("driver_roster", "Which driver is rostered on which duty on a date.", "date"),
    ("driver_leave", "Approved leave for a driver.", "driver"),
    ("driver_training", "Training records and licence expiry of a driver.", "driver"),
    ("payroll_period", "Payroll status for a pay period.", "period"),
    ("fuel_card_spend", "Fuel card transactions for a depot this month.", "depot"),
    ("charger_availability", "Free charging points at a depot right now.", "depot"),
    ("energy_price", "Contracted electricity price for a depot.", "depot"),
    ("cleaning_schedule", "Interior cleaning schedule for a depot.", "depot"),
    ("lost_property", "Items handed in on buses, by description.", "description"),
    ("passenger_complaints", "Complaints from passengers, by route or topic.", "topic"),
    ("cctv_request", "Status of a request for on-board camera footage.", "request_id"),
    ("insurance_claims", "Open insurance claims after collisions.", "claim_id"),
    ("accident_reports", "Accident reports filed by drivers.", "report_id"),
    ("supplier_contracts", "Framework contracts with a supplier.", "supplier"),
    ("invoice_status", "Payment status of a supplier invoice.", "invoice_no"),
    ("purchase_orders", "Purchase orders raised by procurement.", "po_number"),
    ("tyre_inspections", "Tyre tread readings from the last depot inspection.", "depot"),
    ("bus_washing_log", "When a bus last went through the wash.", "fleet_no"),
    ("advertising_contracts", "Advertising wraps booked on the fleet.", "advertiser"),
    ("school_contracts", "Contracted school trips for a term.", "school"),
    ("event_services", "Extra buses planned for a public event.", "event"),
    ("staff_directory", "Phone number and role of a staff member.", "name"),
    ("it_helpdesk_tickets", "Office IT helpdesk tickets.", "ticket_id"),
    ("meeting_rooms", "Free meeting rooms at head office.", "date"),
    ("weather_forecast", "Weather forecast for a city.", "city"),
    ("road_works", "Planned road works affecting bus routes.", "city"),
    ("punctuality_reports", "Punctuality figures reported to the transport authority.", "month"),
    ("emissions_report", "CO2 emissions of a depot for a month.", "depot"),
    ("depot_access_badges", "Access badges issued for a depot.", "depot"),
    ("parking_permits", "Staff parking permits at a depot.", "depot"),
    ("uniform_orders", "Uniform orders for drivers.", "driver"),
    ("safety_incidents", "Workplace safety incidents in depots.", "depot"),
    ("vacancies", "Open job vacancies.", "role"),
    ("helpline_calls", "Calls to the passenger helpline.", "topic"),
    ("social_media_mentions", "Mentions of the company on social media.", "topic"),
    ("department_budget", "Budget and spend for a department.", "department"),
    ("board_decisions", "Decisions of the management board.", "topic"),
    ("tender_documents", "Documents of an open tender.", "tender"),
    ("bus_position", "GPS position of a bus right now.", "fleet_no"),
    ("delay_alerts", "Live delay alerts on a route.", "route"),
    ("signal_priority", "Bus priority settings at a junction.", "junction"),
    ("bike_rack_bookings", "Bike rack bookings on intercity routes.", "route"),
    ("assistance_requests", "Assistance requests from passengers with a disability.", "request_id"),
    ("survey_results", "Passenger satisfaction survey results for a route.", "route"),
    ("holiday_timetables", "Timetable changes for a public holiday.", "holiday"),
    ("driver_breaks", "Scheduled break locations for drivers on a route.", "route"),
]


def other_system(name, description, arg):
    return StructuredTool.from_function(
        func=lambda **kw: f"no records for {kw[arg]}", name=name, description=description,
        args_schema=create_model(name, **{arg: (str, ...)}))


OTHER_TOOLS = [other_system(*spec) for spec in OTHER]
print("other tools:", len(OTHER_TOOLS))

other tools: 52


We use six questions over the desk's data, each with the facts a correct
answer contains. `Meter` counts model calls and tokens through a callback,
and `correct` checks the facts. A `|` separates accepted alternatives.
F-CD-101 in the last question is a CD-12 fault code, so a correct answer
names the model as well as the mechanic ; an answer that treats the code as
a fleet number fails.

In [5]:
from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

QUESTIONS = [
    ("Is part CE-DA-03 in stock?", ["0|out of stock|not in stock|none"]),
    ("What firmware runs on bus CE12-2023-0342?", ["5.1"]),
    ("Which mechanics are certified for the CD-12?", ["nora dalen", "ivo rietman"]),
    ("Bus CE12-2021-0117 shows F-CE-110. What part do I need and is it in stock?",
     ["ce-cf-02|cooling fan", "6|six"]),
    ("What fixed F-RR-120 on earlier tickets?", ["rr-bp-03|auxiliary battery pack"]),
    ("Which mechanic in region Zuid can fix F-CD-101 at Broekdam Depot?", ["nora dalen", "cd-12"]),
]

## a · one-agent-limits

We build one agent with `n` tools : the eight core tools plus the
first `n - 8` others, shuffled so the core tools are not all at the front.
It asks every question once and prints a line per question. A detour is a
call to one of the other systems' tools.

In [6]:
import random

from langchain.agents import create_agent

PROMPT = "You are the service desk assistant for Veenhaven Bus. Answer from the tools only. Be brief."
CORE_NAMES = {t.name for t in CORE}
results = []


def measure(n):
    tools = CORE + OTHER_TOOLS[:n - len(CORE)]
    random.Random(n).shuffle(tools)
    agent = create_agent(llm, tools=tools, system_prompt=PROMPT)
    row = {"tools": n, "correct": 0, "calls": 0, "input": 0, "output": 0, "detours": 0}
    for question, expected in QUESTIONS:
        meter = Meter()
        messages = agent.invoke({"messages": [{"role": "user", "content": question}]},
                                {"callbacks": [meter]})["messages"]
        used = [m.name for m in messages if m.type == "tool"]
        ok = correct(messages[-1].content, expected)
        detours = sum(name not in CORE_NAMES for name in used)
        for key, value in [("correct", ok), ("calls", meter.calls), ("input", meter.input),
                           ("output", meter.output), ("detours", detours)]:
            row[key] += value
        print(f"{'ok' if ok else 'WRONG':5} {meter.calls} calls {meter.input:>7,} in  {used}")
        print("answer:", messages[-1].content)
    results.append(row)
    print(f"{n} tools : {row['correct']}/{len(QUESTIONS)} correct, {row['calls']} calls, "
          f"{row['input']:,} input tokens, {row['detours']} detours")

First the eight core tools alone. This is the baseline : every tool is
relevant, and the tool list is short.

In [7]:
measure(8)

ok    2 calls     739 in  ['part_stock']
answer: No. CE-DA-03 (door actuator) is out of stock at Rietpoort.


ok    2 calls     751 in  ['bus_info']
answer: Bus CE12-2023-0342 runs firmware version 5.1.


ok    2 calls     754 in  ['mechanics_for_model']
answer: The mechanics certified for the CD-12 are Nora Dalen (Zuid) and Ivo Rietman (Noord).


ok    3 calls   2,901 in  ['bus_info', 'search_manuals', 'search_tickets', 'part_stock']
answer: F-CE-110 indicates traction system over-temperature. Check the traction cooling fan and replace it if airflow is below 80%. The part is **CE-CF-02**, the traction cooling fan; **6 are in stock at Rietpoort**.


ok    2 calls   1,152 in  ['search_tickets']
answer: Earlier tickets resolved F-RR-120 by replacing part **RR-BP-03**; the door controller then passed its self-test. Repairs took **1.5–6 hours** across the tickets returned.


ok    4 calls   1,680 in  ['bus_info', 'depot_info', 'mechanics_for_model']
answer: I couldn’t verify the bus model: the system says **F-CD-101** does not exist. If it’s a **CD-12**, **Nora Dalen** is certified and works in region Zuid.
8 tools : 6/6 correct, 15 calls, 7,977 input tokens, 0 detours


Twenty tools : twelve of them belong to other systems.

In [8]:
measure(20)

ok    2 calls   1,317 in  ['part_stock']
answer: No. CE-DA-03 (door actuator) is out of stock at Rietpoort.


ok    2 calls   1,329 in  ['bus_info']
answer: Bus CE12-2023-0342 runs firmware version 5.1.


ok    2 calls   1,332 in  ['mechanics_for_model']
answer: Nora Dalen (Zuid) and Ivo Rietman (Noord) are certified for the CD-12.


ok    3 calls   3,766 in  ['bus_info', 'search_tickets', 'search_manuals', 'part_stock']
answer: F-CE-110 indicates a traction system over-temperature. Check the traction cooling fan and replace it if airflow is below 80%. The part is **CE-CF-02 (traction cooling fan)**; **6 are in stock at Rietpoort**.


ok    2 calls   1,730 in  ['search_tickets']
answer: Earlier F-RR-120 tickets were fixed by replacing part **RR-BP-03**; the door controller passed its self-test afterward.


WRONG 3 calls   2,007 in  ['bus_info', 'depot_info']
answer: I couldn’t find bus **F-CD-101** in the fleet records, so I can’t identify a mechanic for it. Broekdam Depot is in Broekdam.
20 tools : 5/6 correct, 14 calls, 11,481 input tokens, 0 detours


Forty tools.

In [9]:
measure(40)

ok    2 calls   2,227 in  ['part_stock']
answer: No. CE-DA-03 (door actuator) has 0 units in stock at Rietpoort.


ok    2 calls   2,239 in  ['bus_info']
answer: Bus CE12-2023-0342 runs firmware version 5.1.


ok    2 calls   2,242 in  ['mechanics_for_model']
answer: Nora Dalen (Zuid) and Ivo Rietman (Noord).


ok    3 calls   5,131 in  ['bus_info', 'search_tickets', 'search_manuals', 'part_stock']
answer: F-CE-110 indicates traction system over-temperature. Check the traction cooling fan and replace it if airflow is below 80%. The fan is part **CE-CF-02**; **6 are in stock** at the Rietpoort depot.


ok    2 calls   2,640 in  ['search_tickets']
answer: Earlier tickets resolved F-RR-120 by replacing part **RR-BP-03**; the door-controller self-test then passed.


ok    6 calls   8,102 in  ['mechanics_for_model', 'bus_info', 'depot_info', 'search_tickets', 'mechanics_for_model']
answer: Nora Dalen is certified for the CD-12 in region Zuid. The available records don’t confirm the bus model at Broekdam Depot.
40 tools : 6/6 correct, 17 calls, 22,581 input tokens, 0 detours


We reach sixty tools : eight repair tools and 52 distractors.

In [10]:
measure(60)

ok    2 calls   3,117 in  ['part_stock']
answer: No. Part CE-DA-03 (door actuator) is out of stock at Rietpoort.


ok    2 calls   3,129 in  ['bus_info']
answer: Bus CE12-2023-0342 runs firmware version 5.1.


ok    2 calls   3,132 in  ['mechanics_for_model']
answer: Certified CD-12 mechanics: Nora Dalen (Zuid) and Ivo Rietman (Noord).


ok    3 calls   5,706 in  ['search_tickets', 'part_stock']
answer: F-CE-110 previously required a **CE-CF-02 traction cooling fan**. **6 are in stock** at the Rietpoort depot.


ok    2 calls   3,530 in  ['search_tickets']
answer: Earlier F-RR-120 tickets were resolved by replacing part **RR-BP-03**; the door controller self-test then passed.


WRONG 4 calls   6,339 in  ['bus_info', 'mechanics_for_model', 'depot_info']
answer: I couldn’t identify a bus or mechanic for **F-CD-101**. The system reports no such bus, and returns no mechanics for that model. Broekdam Depot is in Broekdam.
60 tools : 5/6 correct, 15 calls, 24,953 input tokens, 0 detours


We compare the four runs, including average input tokens per call. The
model receives the full tool list on every call, including unused tools.

In [11]:
print(f"{'tools':>5} {'correct':>8} {'calls':>6} {'input':>8} {'per call':>9} {'output':>7} {'detours':>8}")
for r in results:
    print(f"{r['tools']:>5} {r['correct']:>6}/{len(QUESTIONS)} {r['calls']:>6} {r['input']:>8,} "
          f"{r['input'] // r['calls']:>9,} {r['output']:>7} {r['detours']:>8}")

tools  correct  calls    input  per call  output  detours
    8      6/6     15    7,977       531     490        0
   20      5/6     14   11,481       820     455        0
   40      6/6     17   22,581     1,328     512        0
   60      5/6     15   24,953     1,663     402        0


We compare correctness, calls and tokens before splitting the agent. More
tool descriptions add input context, but correctness need not decrease.
One run per question is a small sample ; the fact check uses text matches,
so we also read each answer for contradictions and unsupported claims.

Read the answers above the table, not only the scores. A text match passes
an answer that names the right part with the wrong label, and a single run
per size cannot separate a trend from noise.

## Things to try

Optional.

- Rename three of the other tools so they overlap with the core ones, for
  example `fleet_register` or `warehouse_stock`, and measure again
- Run `measure(60)` with `make_llm(model="xiaomi/mimo-v2.6-pro")` in place
  of `llm`
- Shorten the docstring of `part_stock` to two words and see which
  questions fail